In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

In [ ]:
def load_df(dataset_name):
    pred_df = pd.concat(
        [pd.read_csv(pred_file) for pred_file in Path(f"./results/pred/stat_exp/").glob(f"*{dataset_name}*.csv")] +
        [pd.read_csv(pred_file) for pred_file in Path(f"./results/pred/rnn_exp/").glob(f"*{dataset_name}*.csv")] + 
        [pd.read_csv(pred_file) for pred_file in Path(f"./results/pred/pretrained/").glob(f"*{dataset_name}*.csv")]
    )

In [ ]:
import torch
from torchmetrics.regression import SymmetricMeanAbsolutePercentageError

from sklearn.metrics import mean_squared_error

error_function = SymmetricMeanAbsolutePercentageError()

WINDOW, CONTEXT = 100, 50

def smape_degradation(y_pred, y_true):
    errors = {}
    for i in range(WINDOW - CONTEXT):
        err_params = (torch.tensor(y_true[f"y_{i}"]), torch.tensor(y_pred[f"y_pred_{i}"]))
        error_i = error_function(*err_params)
        errors[f"t+{i}"] = error_i.item() * 100 # percentage
        # print(error_i)
    return errors


def rmse_degradation(y_pred, y_true):
    errors = {}
    for i in range(WINDOW - CONTEXT):
        key_true = f"y_{i}"
        key_pred = f"y_pred_{i}"
        
        try:
            true_vals = y_true[key_true]
            pred_vals = y_pred[key_pred]
            
            if not np.all(np.isfinite(true_vals)) or not np.all(np.isfinite(pred_vals)):
                raise ValueError(f"Non-finite values encountered at {y_pred['ts', 'model'].unique()} t+{i}:\n"
                                 f"true_vals: {true_vals.values}\n"
                                 f"pred_vals: {pred_vals.values}")
            
            error_i = mean_squared_error(true_vals, pred_vals, squared=True)
            errors[f"t+{i}"] = error_i.item() * 100  # percentage scale
            
        except Exception as e:
            print(f"[ERROR] Failed at t+{i}")
            print(f"Exception: {e}")
            print(f"true_vals: {true_vals.values if 'true_vals' in locals() else 'N/A'}")
            print(f"pred_vals: {pred_vals.values if 'pred_vals' in locals() else 'N/A'}")
            errors[f"t+{i}"] = np.nan  # or optionally skip this entry

    return errors

def degradation(err_fn, y_pred, y_true):
    errors = {}
    for i in range(WINDOW - CONTEXT):
        # print(f"Computing error for step {i}")
        err_params = (y_true[f"y_{i}"], y_pred[f"y_pred_{i}"])
        # print(err_params)
        error_function = globals()[err_fn]
        error_i = error_function(*err_params)
        errors[f"t+{i}"] = error_i
        # print(error_i)
    return errors

In [ ]:
datasets = ["WSD_3k", "AFD"]

In [ ]:
for dataset_name in datasets:

    pred_df = load_df(dataset_name=dataset_name)

    res = []

    for (dataset, ts, model), sub_df in tqdm(pred_df.groupby(["dataset", "ts", "model"]), desc="Computing errors"):
        res_block = {"dataset": dataset, "ts": ts, "model": model, "model": model}

        y_pred = sub_df[[col for col in sub_df.columns if col.startswith("y_pred")]]
        y_true = sub_df[[col for col in sub_df.columns if col.startswith("y_") and "pred" not in col]]

        

        degradation_errors = smape_degradation(y_pred = y_pred, y_true = y_true)
        # degradation_errors = rmse_degradation(y_pred = y_pred, y_true = y_true)
        
        res_block.update(degradation_errors)

        res.append(res_block)

    res_df = pd.DataFrame(res)

    res_df.to_csv(f"./results/{dataset_name}_metrics.csv", index=False)